# DAG DSL → Graphviz

Build a model with the SKaiNET DAG DSL and return it from a cell to see it rendered as inline SVG. The DOT → SVG step runs entirely on the JVM kernel via the bundled Graphviz wasm artifact — no JS, no CDN, no browser-side runtime.

The notebook integration registers a `GraphProgram.asDot()` helper that lowers the symbolic DAG to a `ComputeGraph` and serializes it with SKaiNET's `toGraphviz` exporter, so you can write `dag { … }.asDot()` instead of chaining the two steps by hand.

**Prerequisite:** run `./gradlew :kotlin-notebook:publishToMavenLocal` from the repo root, then restart the kernel.

In [ ]:
USE {
    repositories {
        mavenLocal()
    }
    dependencies {
        implementation("sk.ainet.app:kotlin-notebook:0.25.0")
    }
}

## Define a tiny conv block

A 3×3 stride-2 convolution over a 224×224 RGB input, biased, followed by `relu`. `dag`, `input`, `parameter`, `constant`, `conv2d`, `relu`, `output`, and `TensorSpec` are auto-imported by the integration — no explicit `import` lines needed in cells. `asDot()` is auto-imported the same way.

In [ ]:
val program = dag {
    val x = input<FP32>("input", TensorSpec("input", listOf(1, 3, 224, 224), "FP32"))

    val w = parameter<FP32, Float>("weight") { shape(64, 3, 3, 3) { ones() } }
    val b = constant<FP32, Float>("bias") { shape(64) { zeros() } }

    val conv = conv2d(x, w, b, stride = 2 to 2, padding = 1 to 1)
    val activated = relu(conv)

    output(activated)
}

program.asDot()

## Inspect the underlying DOT

`asDot()` returns a `Dot(val source: String)` — handy if you want to paste the text into a separate Graphviz tool or commit it as a fixture.

In [ ]:
println(program.asDot().source)

## Vertical layout

`asDot(rankdir = "TB")` flips the graph from left-to-right to top-to-bottom — easier to read for deep stacks.

In [ ]:
renderDot(program.asDot(rankdir = "TB")) {
    maxHeight = "720px"
}